In [0]:
PROJECT_NAME = "retail-lakehouse-demo"
ENVIRONMENT = "dev"

PREFERRED_CATALOG = "retail_lakehouse"

SCHEMAS = [
    "landing",
    "bronze",
    "silver",
    "gold",
    "audit",
    "quarantine"
]

print(f"Project     : {PROJECT_NAME}")
print(f"Environment : {ENVIRONMENT}")
print(f"Catalog     : {PREFERRED_CATALOG}")

In [0]:
environment_info = spark.sql("""
    SELECT
        current_user() AS current_user,
        current_catalog() AS current_catalog,
        current_schema() AS current_schema
""")

environment_info.show(truncate=False)

In [0]:
spark.sql("SHOW CATALOGS").show(truncate=False)

In [0]:
from pyspark.sql.utils import AnalysisException

preferred_catalog = "retail_lakehouse"

try:
    spark.sql(f"""
        CREATE CATALOG IF NOT EXISTS {preferred_catalog}
        COMMENT 'Retail lakehouse demo catalog'
    """)

    target_catalog = preferred_catalog
    print(f"Using custom catalog: {target_catalog}")

except Exception as exc:
    target_catalog = spark.sql(
        "SELECT current_catalog() AS catalog"
    ).first()["catalog"]

    print("Custom catalog creation was not available.")
    print(f"Using existing catalog instead: {target_catalog}")
    print(f"Details: {str(exc)[:300]}")

In [0]:
schemas = [
    "landing",
    "bronze",
    "silver",
    "gold",
    "audit",
    "quarantine"
]

for schema_name in schemas:
    spark.sql(f"""
        CREATE SCHEMA IF NOT EXISTS {target_catalog}.{schema_name}
        COMMENT 'Schema for the {schema_name} layer of the retail lakehouse'
    """)

    print(f"Schema ready: {target_catalog}.{schema_name}")

In [0]:
spark.sql(
    f"SHOW SCHEMAS IN {target_catalog}"
).show(truncate=False)

In [0]:
from pyspark.sql import Row
from pyspark.sql import functions as F

config_records = [
    Row(
        config_key="project_name",
        config_value="retail-lakehouse-demo",
        description="Portfolio project name"
    ),
    Row(
        config_key="environment",
        config_value="dev",
        description="Current execution environment"
    ),
    Row(
        config_key="source_system",
        config_value="retail_pos",
        description="Primary source system"
    ),
    Row(
        config_key="default_currency",
        config_value="INR",
        description="Default reporting currency"
    ),
    Row(
        config_key="timezone",
        config_value="Asia/Kolkata",
        description="Business reporting timezone"
    )
]

config_df = (
    spark.createDataFrame(config_records)
    .withColumn("created_timestamp", F.current_timestamp())
)

config_table = f"{target_catalog}.audit.pipeline_configuration"

(
    config_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(config_table)
)

print(f"Created configuration table: {config_table}")

In [0]:
spark.table(config_table).show(truncate=False)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {target_catalog}.audit.pipeline_runs
(
    pipeline_run_id STRING,
    pipeline_name STRING,
    notebook_name STRING,
    source_system STRING,
    source_file_name STRING,
    source_file_path STRING,
    layer STRING,
    load_type STRING,
    run_status STRING,
    records_read BIGINT,
    records_written BIGINT,
    records_rejected BIGINT,
    start_timestamp TIMESTAMP,
    end_timestamp TIMESTAMP,
    error_message STRING,
    created_timestamp TIMESTAMP
)
USING DELTA
COMMENT 'Execution audit records for retail data pipelines'
""")

In [0]:
spark.sql(
    f"DESCRIBE TABLE {target_catalog}.audit.pipeline_runs"
).show(truncate=False)